# 环节 07 · Agent 落地形态演示

纯 Python 标准库，零依赖。把五种形态的"最小隔离要求"变成可打分的矩阵：

1. 要求矩阵（哪种形态必须要什么）；
2. 给定配置打分，输出缺口；
3. 同一越界动作在不同形态下的结果。

> 产品细节为 **2026-09 快照**，引用前回官网核。

In [ ]:
# §1 五形态最小隔离要求矩阵
FORMS = ["编码Agent", "代码解释器", "评测沙箱", "ComputerUse", "FaaS"]
REQS = {
    "FS隔离":        {"编码Agent": 2, "代码解释器": 2, "评测沙箱": 2, "ComputerUse": 2, "FaaS": 2},
    "网络默认关":    {"编码Agent": 2, "代码解释器": 2, "评测沙箱": 2, "ComputerUse": 2, "FaaS": 1},
    "资源配额":      {"编码Agent": 2, "代码解释器": 2, "评测沙箱": 2, "ComputerUse": 1, "FaaS": 2},
    "凭证不落沙箱":  {"编码Agent": 2, "代码解释器": 2, "评测沙箱": 0, "ComputerUse": 2, "FaaS": 2},
    "每次干净起跑":  {"编码Agent": 1, "代码解释器": 1, "评测沙箱": 2, "ComputerUse": 1, "FaaS": 1},
    "独立内核":      {"编码Agent": 1, "代码解释器": 1, "评测沙箱": 1, "ComputerUse": 1, "FaaS": 1},
    "防作弊":        {"编码Agent": 0, "代码解释器": 0, "评测沙箱": 2, "ComputerUse": 0, "FaaS": 0},
}
LEVEL = {0: "不适用", 1: "建议", 2: "必须"}

print(f"{'要求':<14}" + "".join(f"{f:<14}" for f in FORMS))
print("-" * 84)
for req, m in REQS.items():
    print(f"{req:<14}" + "".join(f"{LEVEL[m[f]]:<14}" for f in FORMS))

In [ ]:
# §2 给一份实际配置打分（2=满足 / 1=部分 / 0=缺失），输出缺口
CONFIG_SCORES = {
    "FS隔离": 2, "网络默认关": 0, "资源配额": 1,
    "凭证不落沙箱": 0, "每次干净起跑": 1, "独立内核": 0, "防作弊": 0,
}


def audit(form):
    gaps, soft = [], []
    for req, m in REQS.items():
        need = m[form]
        if need == 0:
            continue
        have = CONFIG_SCORES[req]
        if have < need:
            (gaps if need == 2 and have == 0 else soft).append(f"{req}(需{LEVEL[need]}/有{have})")
    return gaps, soft


gaps, soft = audit("编码Agent")
print("形态 = 编码 Agent")
print("  硬缺口(必须但缺失):", gaps or "无")
print("  软缺口(建议但不足):", soft or "无")
print()
print("→ 网络默认关 + 凭证不落沙箱 是这类场景最常被漏掉、也最致命的两个")

In [ ]:
# §3 同一越界动作在不同形态下的结果
ACTIONS = [
    ("读 ~/.ssh/id_rsa",           {"编码Agent": "被保护路径拒绝",
                                    "代码解释器": "容器内无此文件",
                                    "评测沙箱": "离线+无此文件",
                                    "ComputerUse": "账号隔离，无宿主密钥",
                                    "FaaS": "容器内无此文件"}),
    ("curl 外传数据",              {"编码Agent": "默认断网/白名单拒绝",
                                    "代码解释器": "建议断网（可能失败）",
                                    "评测沙箱": "必须离线 → 拒绝",
                                    "ComputerUse": "限目标站点",
                                    "FaaS": "视配置"}),
    ("fork 炸弹",                  {"编码Agent": "pids.max 拒绝",
                                    "代码解释器": "pids.max 拒绝",
                                    "评测沙箱": "pids.max 拒绝",
                                    "ComputerUse": "建议加配额",
                                    "FaaS": "配额拒绝"}),
    ("挂 docker.sock 起特权容器",  {"编码Agent": "socket 默认禁 → 拒绝",
                                    "代码解释器": "容器内无此 socket",
                                    "评测沙箱": "无此 socket",
                                    "ComputerUse": "不在沙箱内，另行隔离",
                                    "FaaS": "无此 socket"}),
]
for act, res in ACTIONS:
    print(f"动作: {act}")
    for form, outcome in res.items():
        print(f"    {form:<14}{outcome}")
    print()

## §4 自测表

| # | 问题 | 答案要点 |
|---|---|---|
| 1 | Claude Code 的沙箱与权限是什么关系？ | 两层正交：权限管批不批，沙箱管批准后碰什么 |
| 2 | 为什么编码 Agent 的沙箱只隔离 Bash？ | 内置文件工具走权限系统；Computer Use 在真实桌面 |
| 3 | Codex 的三档网络策略？ | `Isolated` / `ProxyOnly` / `FullAccess` |
| 4 | 评测沙箱与代码解释器的最大差别？ | 防作弊 + 离线 + 答案不可读 + 一次性 |
| 5 | gVisor / Firecracker 各举一个产品？ | Modal(gVisor) / E2B(Firecracker) |
| 6 | "用 Docker 跑用户代码"够吗？ | 单租户防误操作够；多租户不可信需换档并补配置 |

**相关长文**：[环节07-Agent落地形态详解.md](./环节07-Agent落地形态详解.md)